In [1]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import numpy as np
import torch
import random
import pickle

In [2]:
config = {
    "seed": 0,
    "cutoff_date": "2020-01-01",
    "test_cutoff_date": "2022-05-01",
    "max_len": 384,
    "batch_size": 1,
    "learning_rate": 1e-4,
    "weight_decay": 0.0,
    "mixed_precision": "bf16",
    "model_config_path": "../working/configs/pairwise.yaml",  # Adjust path as needed
    "epochs": 10,
    "cos_epoch": 5,
    "loss_power_scale": 1.0,
    "max_cycles": 1,
    "grad_clip": 0.1,
    "gradient_accumulation_steps": 1,
    "d_clamp": 30,
    "max_len_filter": 9999999,
    "structural_violation_epoch": 50,
    "balance_weight": False,
}

In [3]:
test_data=pd.read_csv("/kaggle/input/stanford-ribonanza-2-rna-folding-in-3-d/test_sequences.csv")

In [4]:
from torch.utils.data import Dataset, DataLoader

class RNADataset(Dataset):
    def __init__(self,data):
        self.data=data
        self.tokens={nt:i for i,nt in enumerate('ACGU')}

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        sequence=[self.tokens[nt] for nt in (self.data.loc[idx,'sequence'])]
        sequence=np.array(sequence)
        sequence=torch.tensor(sequence)




        return {'sequence':sequence}

In [5]:
test_dataset=RNADataset(test_data)
test_dataset[0]

{'sequence': tensor([2, 2, 2, 2, 2, 1, 1, 0, 1, 0, 2, 1, 0, 2, 0, 0, 2, 1, 2, 3, 3, 1, 0, 1,
         2, 3, 1, 2, 1, 0, 2, 1, 1, 1, 1, 3, 2, 3, 1, 0, 2, 1, 1, 0, 3, 3, 2, 1,
         0, 1, 3, 1, 1, 2, 2, 1, 3, 2, 1, 2, 0, 0, 3, 3, 1, 3, 2, 1, 3])}

In [6]:
import sys

sys.path.append("/kaggle/input/ribonanzanet2d-final")


from Network import *
import yaml



class Config:
    def __init__(self, **entries):
        self.__dict__.update(entries)
        self.entries=entries

    def print(self):
        print(self.entries)

def load_config_from_yaml(file_path):
    with open(file_path, 'r') as file:
        config = yaml.safe_load(file)
    return Config(**config)

class finetuned_RibonanzaNet(RibonanzaNet):
    def __init__(self, config, pretrained=False):
        config.dropout=0.2
        super(finetuned_RibonanzaNet, self).__init__(config)
        if pretrained:
            self.load_state_dict(torch.load("/kaggle/input/ribonanzanet-weights/RibonanzaNet.pt",map_location='cpu'))
        # self.ct_predictor=nn.Sequential(nn.Linear(64,256),
        #                                 nn.ReLU(),
        #                                 nn.Linear(256,64),
        #                                 nn.ReLU(),
        #                                 nn.Linear(64,1)) 
        self.dropout=nn.Dropout(0.0)
        self.xyz_predictor=nn.Linear(256,3)

    def forward(self,src):
        
        #with torch.no_grad():
        sequence_features, pairwise_features=self.get_embeddings(src, torch.ones_like(src).long().to(src.device))

        xyz=self.xyz_predictor(sequence_features)

        return xyz

In [7]:
model = finetuned_RibonanzaNet(
    load_config_from_yaml("/kaggle/input/ribonanzanet2d-final/configs/pairwise.yaml"), 
    pretrained=False
).cuda()

model.load_state_dict(torch.load("/kaggle/input/ribonanzanet-3d-finetune/RibonanzaNet-3D.pt", weights_only=True))


constructing 9 ConvTransformerEncoderLayers


<All keys matched successfully>

In [8]:
state_dict = torch.load("/kaggle/input/ribonanzanet-3d-finetune/RibonanzaNet-3D.pt", weights_only=True)
model.load_state_dict(state_dict)


<All keys matched successfully>

In [9]:
test_dataset[0]['sequence'].shape

torch.Size([69])

In [10]:
model.eval()
preds = []

for idx in range(len(test_dataset)):
    src = test_dataset[idx]['sequence'].long().unsqueeze(0).cuda()

    tmp = []

    # Dropout activado manualmente para estimar incertidumbre
    model.dropout.train()
    for _ in range(4):
        with torch.no_grad():
            xyz = model(src).squeeze()
        tmp.append(xyz.cpu().numpy())

    # Predicción determinista
    model.eval()
    with torch.no_grad():
        xyz = model(src).squeeze()
    tmp.append(xyz.cpu().numpy())

    tmp = np.stack(tmp, 0)  # (5, L, 3)
    preds.append(tmp)


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


In [11]:
tmp.shape

(5, 118, 3)

In [12]:
preds[0]

array([[[26.907227 , -9.046555 , -2.278727 ],
        [28.874458 , -4.2607913,  4.04007  ],
        [30.631533 , -1.3104306,  4.6991277],
        ...,
        [-5.5295033, 33.749996 , 23.419554 ],
        [ 0.7620195, 25.552517 , 27.747623 ],
        [ 7.3068843, 24.760235 , 21.798702 ]],

       [[26.907227 , -9.046555 , -2.278727 ],
        [28.874458 , -4.2607913,  4.04007  ],
        [30.631533 , -1.3104306,  4.6991277],
        ...,
        [-5.5295033, 33.749996 , 23.419554 ],
        [ 0.7620195, 25.552517 , 27.747623 ],
        [ 7.3068843, 24.760235 , 21.798702 ]],

       [[26.907227 , -9.046555 , -2.278727 ],
        [28.874458 , -4.2607913,  4.04007  ],
        [30.631533 , -1.3104306,  4.6991277],
        ...,
        [-5.5295033, 33.749996 , 23.419554 ],
        [ 0.7620195, 25.552517 , 27.747623 ],
        [ 7.3068843, 24.760235 , 21.798702 ]],

       [[26.907227 , -9.046555 , -2.278727 ],
        [28.874458 , -4.2607913,  4.04007  ],
        [30.631533 , -1.3104306,  4

In [13]:
preds[7][0].shape

(720, 3)

In [14]:
import plotly.graph_objects as go
import numpy as np

# Example: Generate an Nx3 matrix

xyz = preds[7][0]  # Replace this with your actual Nx3 data
N = len(xyz)

# Extract columns
x, y, z = xyz[:, 0], xyz[:, 1], xyz[:, 2]

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=x, y=y, z=z,
    mode='markers',
    marker=dict(
        size=5,
        color=z,  # Coloring based on z-value
        colorscale='Viridis',  # Choose a colorscale
        opacity=0.8
    )
)])

# Customize layout
fig.update_layout(
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z"
    ),
    title="3D Scatter Plot"
)

# Show figure
fig.show(renderer='iframe')


In [15]:
ID=[]
resname=[]
resid=[]
x=[]
y=[]
z=[]

data=[]

for i in range(len(test_data)):
    #print(test_data.loc[i])

    
    for j in range(len(test_data.loc[i,'sequence'])):
        # ID.append(test_data.loc[i,'sequence_id']+f"_{j+1}")
        # resname.append(test_data.loc[i,'sequence'][j])
        # resid.append(j+1) # 1 indexed
        row=[test_data.loc[i,'target_id']+f"_{j+1}",
             test_data.loc[i,'sequence'][j],
             j+1]

        for k in range(5):
            for kk in range(3):
                row.append(preds[i][k][j][kk])
        data.append(row)

columns=['ID','resname','resid']
for i in range(1,6):
    columns+=[f"x_{i}"]
    columns+=[f"y_{i}"]
    columns+=[f"z_{i}"]


submission=pd.DataFrame(data,columns=columns)


submission
submission.to_csv('submission.csv',index=False)

In [16]:
submission

,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,R1107_1,G,1,26.907227,-9.046555,-2.278727,26.907227,-9.046555,-2.278727,26.907227,-9.046555,-2.278727,26.907227,-9.046555,-2.278727,26.907227,-9.046555,-2.278727
1,R1107_2,G,2,28.874458,-4.260791,4.040070,28.874458,-4.260791,4.040070,28.874458,-4.260791,4.040070,28.874458,-4.260791,4.040070,28.874458,-4.260791,4.040070
2,R1107_3,G,3,30.631533,-1.310431,4.699128,30.631533,-1.310431,4.699128,30.631533,-1.310431,4.699128,30.631533,-1.310431,4.699128,30.631533,-1.310431,4.699128
3,R1107_4,G,4,31.214788,1.479454,6.515719,31.214788,1.479454,6.515719,31.214788,1.479454,6.515719,31.214788,1.479454,6.515719,31.214788,1.479454,6.515719
4,R1107_5,G,5,29.375944,1.194736,5.723046,29.375944,1.194736,5.723046,29.375944,1.194736,5.723046,29.375944,1.194736,5.723046,29.375944,1.194736,5.723046
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2510,R1190_114,U,114,18.383455,18.752460,31.582420,18.383455,18.752460,31.582420,18.383455,18.752460,31.582420,18.383455,18.752460,31.582420,18.383455,18.752460,31.582420
2511,R1190_115,U,115,16.322798,17.701756,32.300236,16.322798,17.701756,32.300236,16.322798,17.701756,32.300236,16.322798,17.701756,32.300236,16.322798,17.701756,32.300236
2512,R1190_116,U,116,15.308034,15.650916,34.541931,15.308034,15.650916,34.541931,15.308034,15.650916,34.541931,15.308034,15.650916,34.541931,15.308034,15.650916,34.541931
2513,R1190_117,U,117,12.111407,16.164152,34.995243,12.111407,16.164152,34.995243,12.111407,16.164152,34.995243,12.111407,16.164152,34.995243,12.111407,16.164152,34.995243
